# Initial statistical EDA draft

**Status:** analysis plan and starter checks prepared. The shared project CSV files have not been supplied for this draft, so there are no empirical findings yet.

My statistical workstream covers data quality, crash distributions, recorded contributing circumstances, and descriptive comparisons between regions. Road preparation and spatial matching are coordinated with the other workstreams.

## Questions and inputs

| Question | Input needed |
|---|---|
| Are identifiers and important fields complete? | Crash CSV and field definitions |
| How many records are available by year and severity? | `YEAR`, `CRASH_SEVERITY_DESCR` |
| Which recorded circumstances are common? | Weather, light, surface, collision and driver-circumstance fields |
| Can regions be compared on the same basis? | Municipality, time coverage and extraction metadata |
| What road attributes are available? | Molly's original processed attributes CSV; road analysis is a later step |

Molly's road export uses `segment_id = GlobalID`. Crash records use `CRASH_NUMB` and `CITY_TOWN_NAME`. Road `City` codes and crash municipality names need an agreed crosswalk before joining.

## Load one shared crash CSV

Run from the repository root or the `notebooks` directory. Install `requirements-eda.txt` in your notebook environment, then set `CRASH_CSV` to one specific shared file. Start with the completed 2023 extract; add the other years after their downloads are validated.

The existing crash download needs five additional fields for the planned circumstances analysis: `WEATH_COND_DESCR`, `AMBNT_LIGHT_DESCR`, `ROAD_SURF_COND_DESCR`, `MANR_COLL_DESCR`, and `DRVR_CNTRB_CIRC_CL`.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CRASH_CSV = None  # Example: "data/raw/crashes_2023.csv"

crashes = None
if CRASH_CSV is None:
    print("Awaiting a shared CSV. No empirical analysis has been run.")
else:
    path = Path(CRASH_CSV)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    crashes = pd.read_csv(path, dtype="string", keep_default_na=False)
    print(f"Loaded {len(crashes):,} input records from {path.name}.")

## Initial schema and quality checks

These checks preserve the input rows. Blank values are counted separately from literal category values such as `Unknown`; source-specific codes require a dictionary review. Repeated crash IDs are flagged for investigation.

Download completeness must also be checked against the source manifest. These checks alone cannot prove that an annual extract contains every record.

In [ ]:
if crashes is None or crashes.empty:
    print("Quality checks pending: no nonempty crash table supplied.")
else:
    expected_fields = [
        "CRASH_NUMB", "YEAR", "CITY_TOWN_NAME", "CRASH_DATETIME",
        "CRASH_SEVERITY_DESCR", "WEATH_COND_DESCR", "AMBNT_LIGHT_DESCR",
        "ROAD_SURF_COND_DESCR", "MANR_COLL_DESCR", "DRVR_CNTRB_CIRC_CL",
    ]
    schema = pd.DataFrame({
        "field": expected_fields,
        "present": [name in crashes.columns for name in expected_fields],
    })
    print(schema.to_string(index=False))

    blank = crashes.isna() | crashes.apply(lambda column: column.str.strip().eq(""))
    quality = pd.DataFrame({
        "missing_or_blank": blank.sum(),
        "percent_missing_or_blank": (blank.mean() * 100).round(1),
    })
    print("\nField completeness:")
    print(quality.to_string())

    if "CRASH_NUMB" in crashes:
        ids = crashes["CRASH_NUMB"].str.strip()
        usable = ids.notna() & ids.ne("")
        print("\nMissing/blank crash IDs:", int((~usable).sum()))
        print("Additional rows with a repeated nonblank crash ID:",
              int(ids[usable].duplicated().sum()))

## Starter record distributions

These are counts of supplied rows. Event identity, geographic scope and extraction completeness still need confirmation. A missing year is not automatically a zero-crash year. Municipal counts are descriptive and do not establish a safety ranking.

Month, weekday and hour summaries are deferred until the CSV timestamp encoding and timezone have been checked. The current export converts epoch milliseconds to text; preserve the original values and agree on the time representation before extending this section.

In [ ]:
if crashes is None or crashes.empty:
    print("Record distributions pending: no nonempty crash table supplied.")
else:
    for field in ["YEAR", "CRASH_SEVERITY_DESCR", "CITY_TOWN_NAME"]:
        if field not in crashes:
            print(f"{field}: field not supplied.")
            continue
        values = crashes[field].str.strip()
        values = values.mask(values.isna() | values.eq(""), "[Missing]")
        counts = values.value_counts().rename_axis(field).reset_index(name="record_count")
        print(f"\n{field} — supplied record counts:")
        print(counts.to_string(index=False))

## Next steps

- Obtain the versioned road CSV from Molly and the completed crash CSV files from Bryan, with extraction metadata.
- Confirm crash-key scope across years, timestamp rules, category definitions and the additional circumstances fields.
- Keep full raw-record EDA separate from the eligible non-intersection cohort. Agree on driveway/ramp, geocoding and facility rules; retain exclusion reasons.
- Add time distributions, circumstances summaries and road profiles after the relevant inputs are checked. Counts of circumstances among crashes do not measure causal effects or travel-exposure-adjusted risk.
- Follow the proposal's outcome-independent geography selection and geographic split rules when moving to the model comparison.

## Reviewed work

- [Road preparation notes — data-processing](https://github.com/RexHsu09/cs506-final-project/blob/877c5e15b620cfd811aee43b33d9045310cd41f7/docs/data_processing.md)
- [Initial crash exploration — Bryan](https://github.com/RexHsu09/cs506-final-project/blob/167c5cdb62e2f79fe8161cb637bb789b1facfd7e/crash_data_exploration.ipynb)

AI assistance was used to prepare this initial draft. The team is responsible for reviewing the code, validating it on the actual datasets, and interpreting results.